In [1]:
import requests
import math
import os
import sys
import platform
from lakehouse import bronze, silver
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql.datasource import DataSource, DataSourceReader, InputPartition
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import functions as F

In [2]:
if platform.system() == 'Windows':
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = SparkSession \
    .builder \
    .appName("Data with Nikk the Greek Spark Session") \
    .master("local[4]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
class SWAPIDataSource(DataSource):
    """
    Implementing the SWAPI API as to https://swapi.dev/documentation

    Name: `SWAPI`

    Schema (depends on the called resource): 

        "people": 'birth_year string, created string, edited string, eye_color string, films array<string>, 
        gender string, hair_color string, height string, homeworld string, mass string, name string, skin_color string, 
        species array<string>, starships array<string>, url string, vehicles array<string>',

        "films": 'characters array<string>, created string, director string, edited string, episode_id bigint, 
        opening_crawl string, planets array<string>, producer string, release_date string, species array<string>, 
        starships array<string>, title string, url string, vehicles array<string>',

        "starships": 'MGLT string, cargo_capacity string, consumables string, cost_in_credits string, created string, 
        crew string, edited string, films array<string>, hyperdrive_rating string, length string, manufacturer string, 
        max_atmosphering_speed string, model string, name string, passengers string, pilots array<string>, starship_class string, url string',

        "vehicles": 'cargo_capacity string, consumables string, cost_in_credits string, created string, crew string, 
        edited string, films array<string>, length string, manufacturer string, max_atmosphering_speed string, model string, 
        name string, passengers string, pilots array<string>, url string, vehicle_class string',

        "species": 'average_height string, average_lifespan string, classification string, created string, designation string, 
        edited string, eye_colors string, films array<string>, hair_colors string, homeworld string, language string, name string, 
        people array<string>, skin_colors string, url string',

        "planets": 'climate string, created string, diameter string, edited string, films array<string>, gravity string, name string, 
        orbital_period string, population string, residents array<string>, rotation_period string, surface_water string, terrain string, url string'

    Examples
    --------
    Register the data source.

    >>> from pyspark_datasources import SWAPIDataSource
    >>> spark.dataSource.register(SWAPIDataSource)

    Load data from the availble resources "people", "films", "starships", "vehicles", "species", "planets"

    >>> spark.read.format("SWAPI").load("planets").show()
 
    +---------+--------------------+--------+--------------------+--------------------+----------+--------+---+
    |  climate|             created|diameter|              edited|               films|   gravity|    name|...|
    +---------+--------------------+--------+--------------------+--------------------+----------+--------+---+
    |     arid|2014-12-09T13:50:...|   10465|2014-12-20T20:58:...|[https://swapi.de...|1 standard|Tatooine|...|
    |...      |...                 |...     |...                 |                 ...|...       |...     |...|
    +---------+--------------------+--------+--------------------+--------------------+----------+--------+---+

    """

    @classmethod
    def name(self):
        return "SWAPI"
    
    def schema(self):
        if self.options["path"] not in ["people", "films", "starships", "vehicles", "species", "planets"]:
            raise Exception(f"Assure that only values in ['people', 'films', 'starships', 'vehicles', 'species', 'planets'] are provided")
        
        schemas = {
            "people": 'birth_year string, created string, edited string, eye_color string, films array<string>, gender string, hair_color string, height string, homeworld string, mass string, name string, skin_color string, species array<string>, starships array<string>, url string, vehicles array<string>',

            "films": 'characters array<string>, created string, director string, edited string, episode_id bigint, opening_crawl string, planets array<string>, producer string, release_date string, species array<string>, starships array<string>, title string, url string, vehicles array<string>',

            "starships": 'MGLT string, cargo_capacity string, consumables string, cost_in_credits string, created string, crew string, edited string, films array<string>, hyperdrive_rating string, length string, manufacturer string, max_atmosphering_speed string, model string, name string, passengers string, pilots array<string>, starship_class string, url string',

            "vehicles": 'cargo_capacity string, consumables string, cost_in_credits string, created string, crew string, edited string, films array<string>, length string, manufacturer string, max_atmosphering_speed string, model string, name string, passengers string, pilots array<string>, url string, vehicle_class string',

            "species": 'average_height string, average_lifespan string, classification string, created string, designation string, edited string, eye_colors string, films array<string>, hair_colors string, homeworld string, language string, name string, people array<string>, skin_colors string, url string',

            "planets": 'climate string, created string, diameter string, edited string, films array<string>, gravity string, name string, orbital_period string, population string, residents array<string>, rotation_period string, surface_water string, terrain string, url string'
        }

        return schemas[self.options["path"]]

    def reader(self, schema):
        return SWAPIDataSourceReader(self.options)


class SWAPIDataSourceReader(DataSourceReader):
    def __init__(self, options):
        self.resource = options["path"]

    def partitions(self):
        query = f"https://swapi.dev/api/{self.resource}/"
        page_size = 10
        total_elements = int(requests.get(query).json()["count"])
        no_pages = math.ceil(total_elements / page_size)
        return [InputPartition(i) for i in range(1, no_pages + 1)]

    def read(self, partition):
        query = f"https://swapi.dev/api/{self.resource}/?page={str(partition.value)}"
        req = requests.get(query)
        data = req.json()["results"]
        for d in data:
            yield Row(**d)

In [6]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [7]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze" 
}

In [8]:
class StarWarsBronze(bronze.BronzeOverwrite):
    def load(self, table):
        spark.dataSource.register(SWAPIDataSource)
        return spark.read.format("SWAPI").load(table)
    
bronze_instance = StarWarsBronze(spark, **options)
bronze_instance.execute_all(["people", "planets"])

In [9]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+----------+--------------------+--------------------+-------------+--------------------+------+----------+------+--------------------+-------+---------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|birth_year|             created|              edited|    eye_color|               films|gender|hair_color|height|           homeworld|   mass|           name|         skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS|
+----------+--------------------+--------------------+-------------+--------------------+------+----------+------+--------------------+-------+---------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|     62BBY|2014-12-19T17:55:...|2014-12-20T21:17:...|        brown|[https://swapi.de...|  male|     black|   183|https://swapi.dev.

In [10]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 60
+--------------------+--------------------+--------+--------------------+--------------------+-------------+--------------+--------------+------------+--------------------+---------------+-------------+--------------------+--------------------+--------------------+
|             climate|             created|diameter|              edited|               films|      gravity|          name|orbital_period|  population|           residents|rotation_period|surface_water|             terrain|                 url|         LH_BronzeTS|
+--------------------+--------------------+--------+--------------------+--------------------+-------------+--------------+--------------+------------+--------------------+---------------+-------------+--------------------+--------------------+--------------------+
|     temperate, arid|2014-12-10T12:47:...|   11370|2014-12-20T20:58:...|[https://swapi.de...| 0.9 standard|      Geonosis|           256|100000000000|[https://swapi.de...|             30| 

In [11]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
    "merge_schema": False, 
}

# 2 Overwrite with execute_one and execute_all

In [12]:
#with load filter and transformation
class StarWarsSilver(silver.SilverOverwrite):
    def filter_load(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("eye_color == 'blue'")
    
    def transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("id", F.split(sdf["url"], "/").getItem(5).cast("int"))
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.execute_one("people")

In [13]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 19
+----------+--------------------+--------------------+---------+--------------------+------+------------+------+--------------------+-------+------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+---+--------------------+
|birth_year|             created|              edited|eye_color|               films|gender|  hair_color|height|           homeworld|   mass|              name|skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS| id|         LH_SilverTS|
+----------+--------------------+--------------------+---------+--------------------+------+------------+------+--------------------+-------+------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+---+--------------------+
|   41.9BBY|2014-12-10T16:20:...|2014-12-20T21:17:...|     blue|[https://swapi.de

In [15]:
#debug
silver_instance.debug_one("people", n=2).show()

+----------+--------------------+--------------------+---------+--------------------+------+------------+------+--------------------+-------+------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+---+
|birth_year|             created|              edited|eye_color|               films|gender|  hair_color|height|           homeworld|   mass|              name|skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS| id|
+----------+--------------------+--------------------+---------+--------------------+------+------------+------+--------------------+-------+------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+---+
|     19BBY|2014-12-09T13:50:...|2014-12-20T21:17:...|     blue|[https://swapi.de...|  male|       blond|   172|https://swapi.dev...|     77|    Luke Skywalk

In [16]:
#without load filter 
silver_instance = silver.SilverOverwrite(spark, **options)
silver_instance.execute_one("people")

In [17]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+----------+--------------------+--------------------+---------+--------------------+-------------+------------+------+--------------------+-------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|birth_year|             created|              edited|eye_color|               films|       gender|  hair_color|height|           homeworld|   mass|                name|      skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS|  id|         LH_SilverTS|
+----------+--------------------+--------------------+---------+--------------------+-------------+------------+------+--------------------+-------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|   41.9BBY|2014-12-10T16:20:...|

In [18]:
#for all tables
silver_instance = silver.SilverOverwrite(spark, **options)
silver_instance.execute_all(["people", "planets"])

In [19]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+----------+--------------------+--------------------+---------+--------------------+------+----------+------+--------------------+-------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|birth_year|             created|              edited|eye_color|               films|gender|hair_color|height|           homeworld|   mass|               name|         skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS|  id|         LH_SilverTS|
+----------+--------------------+--------------------+---------+--------------------+------+----------+------+--------------------+-------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|     82BBY|2014-12-20T15:59:...|2014-12-20T21:17:...|

In [20]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 60
+--------------------+--------------------+--------+--------------------+--------------------+-------------+-----------+--------------+-----------+--------------------+---------------+-------------+--------------------+--------------------+--------------------+--------------------+
|             climate|             created|diameter|              edited|               films|      gravity|       name|orbital_period| population|           residents|rotation_period|surface_water|             terrain|                 url|         LH_BronzeTS|         LH_SilverTS|
+--------------------+--------------------+--------+--------------------+--------------------+-------------+-----------+--------------+-----------+--------------------+---------------+-------------+--------------------+--------------------+--------------------+--------------------+
|            polluted|2014-12-10T16:26:...|   13490|2014-12-20T20:58:...|                  []|   1 standard|     Eriadu|           360|220

# 3 Replace Where

In [21]:
class StarWarsSilver(silver.SilverReplaceWhere): 
    def filter_load(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.where("edited == '2014-12-20T21:17:50.403000Z'")
    
    def get_replace_condition(self, sdf: DataFrame, table: str) -> str:
        return "edited == '2014-12-20T21:17:50.403000Z'"
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.execute_one("people")

In [22]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 82
+----------+--------------------+--------------------+---------+--------------------+------+----------+------+--------------------+-------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|birth_year|             created|              edited|eye_color|               films|gender|hair_color|height|           homeworld|   mass|               name|         skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS|  id|         LH_SilverTS|
+----------+--------------------+--------------------+---------+--------------------+------+----------+------+--------------------+-------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|     82BBY|2014-12-20T15:59:...|2014-12-20T21:17:...|

# 4 Append

In [23]:
silver_instance = silver.SilverAppend(spark, **options)
silver_instance.execute_one("people")

In [24]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 164
+----------+--------------------+--------------------+---------+--------------------+------+----------+------+--------------------+-------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|birth_year|             created|              edited|eye_color|               films|gender|hair_color|height|           homeworld|   mass|               name|         skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS|  id|         LH_SilverTS|
+----------+--------------------+--------------------+---------+--------------------+------+----------+------+--------------------+-------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----+--------------------+
|     82BBY|2014-12-20T15:59:...|2014-12-20T21:17:...

# 5 Merge

In [25]:
class StarWarsSilver(silver.SilverMerge): 
    def transform(self, sdf: DataFrame, table: str) -> DataFrame:
        return sdf.withColumn("id", F.split(sdf["url"], "/").getItem(5).cast("int"))
      
    def get_delta_merge_builder(self, sdf: DataFrame, delta_table: DeltaTable) -> DeltaMergeBuilder:
        merge_condition = "target.url = source.url" 
        builder = delta_table.alias("target").merge(sdf.alias("source"), merge_condition)
        builder = builder.whenMatchedUpdateAll()
        builder = builder.whenNotMatchedInsertAll()
        return builder
    
silver_instance = StarWarsSilver(spark, **options)
silver_instance.execute_one("people")

In [26]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {sdf.count()}")
sdf.show()

No. Rows: 164
+----------+--------------------+--------------------+---------+--------------------+-------------+-------------+------+--------------------+-------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+--------------------+
|birth_year|             created|              edited|eye_color|               films|       gender|   hair_color|height|           homeworld|   mass|                name|      skin_color|             species|           starships|                 url|            vehicles|         LH_BronzeTS| id|         LH_SilverTS|
+----------+--------------------+--------------------+---------+--------------------+-------------+-------------+------+--------------------+-------+--------------------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+---+--------------------+
|     19BBY|2014-12-09T13:50:...

# 6 Clean Up

In [27]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")

DataFrame[]